# ¿Influye la financiación recibida en la supervivencia y el éxito de las startups?
### Un análisis exploratorio de datos (EDA) con Python

## 1. Introducción

Una *startup* es una empresa joven que intenta crecer rápido. Para crecer, casi todas necesitan dinero de inversores. A ese dinero se le llama **financiación**.

La pregunta de este trabajo es sencilla:

> **¿Las startups que reciben más dinero duran más y tienen más éxito?**

Para responderla vamos a mirar los datos de miles de startups reales y a buscar si hay una relación entre *cuánto dinero recibieron* y *qué les pasó después*.

## 2. Qué significan las palabras clave

- **Financiación**: el dinero total que ha recibido la startup (en dólares).
- **Supervivencia**: una startup *sobrevive* si **no ha cerrado**. Es decir, si su estado es `operating` (sigue funcionando) o `acquired` (ha sido comprada por otra empresa).
- **Éxito**: lo medimos como que la startup haya sido **comprada** (`acquired`). No es una definición perfecta, pero es la que permiten los datos.
- **Mercado**: el sector al que pertenece la startup (software, biotecnología, juegos...).

## 3. Los datos

Usamos el conjunto de datos **Crunchbase** (`investments_VC.csv`), descargado de Kaggle. Cada fila es una startup y tiene datos como su mercado, su país, el año en que nació, cuánto dinero ha recibido y su estado actual.

## 4. Cómo está organizado el trabajo

1. Preparar los datos (limpieza y valores que faltan)
2. Mirar los datos de forma general
3. Comparar financiación y supervivencia
4. Comprobar con pruebas estadísticas si la relación es real
5. Estudiar los mercados
6. Conclusiones

## 5. Preparación

### 5.1 Herramientas que usamos

Usamos solo cuatro librerías de Python:

- `numpy`: para hacer cálculos con números.
- `pandas`: para trabajar con tablas de datos.
- `matplotlib` y `seaborn`: para hacer gráficas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.options.display.float_format = "{:,.2f}".format

### 5.2 Cargar los datos

Descargamos el archivo de Kaggle con `kagglehub` y lo leemos con `pandas`. Usamos `encoding="latin-1"` porque el archivo tiene letras con tildes y otros símbolos especiales.

In [ ]:
import kagglehub
path = kagglehub.dataset_download("arindam235/startup-investments-crunchbase")
df = pd.read_csv(f"{path}/investments_VC.csv", encoding="latin-1")
print(df.shape)
df.head(3)

### 5.3 Limpiar los datos

Los datos reales siempre vienen "sucios". Aquí arreglamos varios problemas:

1. **Los nombres de las columnas** tienen espacios (por ejemplo `" market "`). Los quitamos y ponemos todo en minúsculas.
2. **Filas vacías y repetidas**: las eliminamos. Cada startup debe aparecer una sola vez.
3. **El texto** tiene espacios sobrantes. Los quitamos para que `"Software "` y `"software"` cuenten como lo mismo.
4. **Los importes de dinero** están guardados como texto, con comas (`"1,500,000"`) y con un guion `-` cuando no hay dato. Los convertimos en números de verdad.
5. **Las fechas** las convertimos a formato de fecha.

In [ ]:
# Nombres de columnas limpios
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Quitar filas vacías y repetidas
n0 = len(df)
df = df.dropna(how="all").drop_duplicates(subset=["permalink"])
print(f"Filas eliminadas (vacías o repetidas): {n0 - len(df)}")

# Limpiar el texto
for c in ["name", "market", "status", "country_code", "region", "city"]:
    df[c] = df[c].astype("string").str.strip()
df["status"] = df["status"].str.lower()
df["market"] = df["market"].str.lower()

def a_numero(s):
    """Convierte texto como ' 1,500,000 ' o '-' en un número (o en hueco si no hay dato)."""
    s = s.astype("string").str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(s.replace({"-": pd.NA, "": pd.NA}), errors="coerce").astype(float)

df["funding_total_usd"] = a_numero(df["funding_total_usd"])

tipos_ronda = ["seed", "venture", "equity_crowdfunding", "undisclosed", "convertible_note",
               "debt_financing", "angel", "grant", "private_equity", "post_ipo_equity",
               "post_ipo_debt", "secondary_market", "product_crowdfunding"]
tipos_ronda = [c for c in tipos_ronda if c in df.columns]
for c in tipos_ronda:
    df[c] = a_numero(df[c])

for c in ["founded_at", "first_funding_at", "last_funding_at"]:
    df[c] = pd.to_datetime(df[c], errors="coerce")
df["founded_year"] = pd.to_numeric(df["founded_year"], errors="coerce").astype(float)
df["funding_rounds"] = pd.to_numeric(df["funding_rounds"], errors="coerce").astype(float)

### 5.4 Valores que faltan (nulos)

En casi todos los datos reales hay huecos. Primero vemos **cuántos faltan** en cada columna.

In [ ]:
nulos = (df.isna().mean().sort_values(ascending=False) * 100).rename("% nulos")
display(nulos.head(12))
display(df[["status", "funding_total_usd", "funding_rounds",
            "founded_year", "country_code", "market"]].isna().sum())

**Qué decidimos hacer con los huecos:**

- Si **no sabemos el estado** (`status`) de una startup, la quitamos. Sin ese dato no podemos saber si sobrevivió.
- Si **falta el mercado o el país**, escribimos `"unknown"` (desconocido).
- Si **falta la financiación**, **no inventamos un valor**. Que falte el dato no significa que la startup no recibiera dinero, solo que no lo sabemos. Si pusiéramos un número inventado, estropearíamos justo la relación que queremos estudiar. Por eso estas startups se quedan fuera solo del análisis de financiación.

In [ ]:
df = df.dropna(subset=["status"]).copy()
df["market"] = df["market"].fillna("unknown")
df["country_code"] = df["country_code"].fillna("unknown")
display(df["status"].value_counts())

### 5.5 Crear columnas nuevas

Creamos columnas que nos ayudarán a responder la pregunta:

- `survived`: vale **1** si la startup no ha cerrado y **0** si cerró.
- `acquired`: vale **1** si la startup fue comprada (nuestro "éxito").
- `us`: vale 1 si la startup es de Estados Unidos.
- `age`: la edad de la startup en años.
- `log_funding`: la financiación en **escala logarítmica**. Hay startups con muy poco dinero y otras con miles de millones. Con el logaritmo, esas diferencias enormes se ven mejor en las gráficas.
- `funding_quartile`: dividimos las startups en **4 grupos del mismo tamaño** según su financiación. Q1 son las que menos dinero recibieron y Q4 las que más.

In [ ]:
REF_YEAR = int(df["last_funding_at"].max().year)   # último año que aparece en los datos
print("Año de referencia:", REF_YEAR)

df["survived"] = (df["status"] != "closed").astype(int)
df["acquired"] = (df["status"] == "acquired").astype(int)
df["us"] = (df["country_code"] == "USA").astype(int)

# Años de fundación imposibles se tratan como dato desconocido
df.loc[(df["founded_year"] < 1990) | (df["founded_year"] > REF_YEAR), "founded_year"] = np.nan
df["age"] = REF_YEAR - df["founded_year"]

# dff = startups con financiación conocida y mayor que 0
dff = df[df["funding_total_usd"] > 0].copy()
dff["log_funding"] = np.log10(dff["funding_total_usd"])
dff["funding_quartile"] = pd.qcut(dff["funding_total_usd"], 4,
                                  labels=["Q1 (menos)", "Q2", "Q3", "Q4 (más)"])
print(f"Startups totales: {len(df):,} | con financiación conocida: {len(dff):,}")

## 6. Primera mirada a los datos

Antes de buscar relaciones, miramos cada variable por separado:

- **Izquierda**: cuántas startups hay en cada estado.
- **Centro**: cómo se reparte el dinero recibido (en escala logarítmica).
- **Derecha**: cuántas rondas de financiación ha tenido cada startup.

Una **ronda** es cada vez que la startup consigue dinero de inversores.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
df["status"].value_counts().plot.bar(ax=ax[0], color="steelblue")
ax[0].set_title("Estado de las startups"); ax[0].set_xlabel("")
sns.histplot(dff["log_funding"], bins=50, ax=ax[1], color="teal")
ax[1].set_title("Financiación total (log10 USD)")
sns.countplot(x=dff["funding_rounds"].clip(upper=10), ax=ax[2], color="coral")
ax[2].set_title("Rondas de financiación (10 = 10 o más)")
plt.tight_layout(); plt.show()

display(dff["funding_total_usd"].describe(percentiles=[.1, .25, .5, .75, .9, .99]))

**Qué se ve aquí:** la mayoría de las startups reciben cantidades pequeñas o medianas y muy pocas reciben cantidades enormes. Por eso la **media** (el promedio) se dispara hacia arriba por unos pocos casos extremos. En este trabajo miramos más la **mediana** (el valor del medio), que es más justa porque no se deja engañar por esos casos extremos.

## 7. Financiación y supervivencia: ¿están relacionadas?

### 7.1 ¿Cuánto dinero recibió cada grupo?

Comparamos la financiación de las startups **cerradas** (`closed`), las que **siguen funcionando** (`operating`) y las **compradas** (`acquired`).

En el gráfico de cajas (*boxplot*), la línea del medio de cada caja es la mediana. Si una caja está más arriba que otra, ese grupo recibió más dinero.

In [ ]:
orden = ["closed", "operating", "acquired"]

display(dff.groupby("status")["funding_total_usd"].agg(["count", "median", "mean"]))

plt.figure(figsize=(8, 5))
sns.boxplot(data=dff[dff["status"].isin(orden)], x="status", y="log_funding",
            order=orden, hue="status", palette="Set2", legend=False)
plt.title("Financiación (log10 USD) según estado"); plt.ylabel("log10(USD)")
plt.show()

### 7.2 ¿Sobreviven más las startups con más dinero?

Aquí usamos los 4 grupos (Q1 a Q4). Para cada grupo calculamos qué porcentaje **sobrevive** y qué porcentaje es **comprado**. Si las barras suben de Q1 a Q4, significa que más dinero va de la mano con más supervivencia.

In [ ]:
tasas = dff.groupby("funding_quartile", observed=True).agg(
    n=("survived", "size"),
    tasa_supervivencia=("survived", "mean"),
    tasa_adquisicion=("acquired", "mean"))
display(tasas.style.format({"tasa_supervivencia": "{:.1%}", "tasa_adquisicion": "{:.1%}"}))

tasas[["tasa_supervivencia", "tasa_adquisicion"]].plot.bar(figsize=(8, 5), rot=0)
plt.title("Supervivencia y adquisición por cuartil de financiación")
plt.ylabel("Proporción"); plt.xlabel("")
plt.show()

### 7.3 Número de rondas y tipo de financiación

Ahora miramos dos cosas más:

- Si **más rondas** de financiación se relacionan con sobrevivir más.
- Si algún **tipo de financiación** (capital semilla, inversores "ángel", capital riesgo...) va con más supervivencia.

In [ ]:
rondas = (dff.assign(rondas=dff["funding_rounds"].clip(upper=8))
          .groupby("rondas").agg(n=("survived", "size"),
                                 supervivencia=("survived", "mean"),
                                 adquisicion=("acquired", "mean")))
display(rondas)

filas = []
for t in [t for t in ["seed", "venture", "angel", "private_equity",
                      "debt_financing", "grant", "convertible_note"] if t in dff.columns]:
    con = dff[dff[t].fillna(0) > 0]
    filas.append({"tipo": t, "n": len(con),
                  "supervivencia": con["survived"].mean(),
                  "adquisicion": con["acquired"].mean()})
display(pd.DataFrame(filas).sort_values("supervivencia", ascending=False))

### 7.4 ¿Y si el motivo es solo la edad?

Una startup **más vieja** ha tenido más tiempo para recibir dinero, pero también más tiempo para cerrar (o para consolidarse). Para no confundirnos, dividimos las startups por **año de fundación** (cohortes) y repetimos la comparación dentro de cada grupo.

En el mapa de calor, los colores más oscuros indican más supervivencia. Si de izquierda a derecha (Q1 a Q4) el color se oscurece **en todas las filas**, la relación se mantiene aunque comparemos startups de la misma edad.

In [ ]:
dff["cohorte"] = pd.cut(dff["founded_year"], [1989, 2000, 2005, 2010, REF_YEAR],
                        labels=["1990-2000", "2001-2005", "2006-2010", f"2011-{REF_YEAR}"])
cohortes = dff.pivot_table(index="cohorte", columns="funding_quartile",
                           values="survived", aggfunc="mean", observed=True)
plt.figure(figsize=(8, 4))
sns.heatmap(cohortes, annot=True, fmt=".0%", cmap="YlGnBu")
plt.title("Supervivencia por cohorte y cuartil de financiación")
plt.show()

## 8. Más gráficas para ver relaciones

### 8.1 Mapa de correlaciones

Una **correlación** es un número entre -1 y 1 que dice si dos cosas suben juntas:

- Cerca de **1**: cuando una sube, la otra también.
- Cerca de **0**: no hay relación clara.
- Cerca de **-1**: cuando una sube, la otra baja.

Usamos la correlación de **Spearman**, que funciona bien aunque haya valores extremos.

In [ ]:
corr = dff[["log_funding", "funding_rounds", "age", "survived", "acquired"]].corr(method="spearman")
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlación de Spearman")
plt.show()

**Como regla rápida:** menos de 0,10 es muy débil, entre 0,10 y 0,30 es débil o moderada, y más de 0,30 es fuerte. Fíjate en la fila `survived`: indica cuánto se relaciona cada variable con sobrevivir.

### 8.2 Pairplot

El *pairplot* dibuja **todas las parejas de variables** a la vez. Cada punto es una startup (usamos una muestra de 2.000 para que sea rápido). En **verde** están las que sobreviven y en **rojo** las que cerraron.

Mira sobre todo la gráfica de arriba a la izquierda (la distribución de `log_funding`): si lo rojo está desplazado hacia la izquierda de lo verde, las cerradas tenían menos dinero.

In [ ]:
muestra = dff[["log_funding", "funding_rounds", "age", "survived"]].dropna()
muestra = muestra.sample(min(2000, len(muestra)), random_state=1)
muestra["estado"] = muestra["survived"].map({1: "sobrevive", 0: "cerrada"})
g = sns.pairplot(muestra.drop(columns="survived"), hue="estado", corner=True,
                 diag_kind="hist", plot_kws={"alpha": 0.4, "s": 12},
                 palette={"sobrevive": "tab:green", "cerrada": "tab:red"})
g.figure.suptitle("Pairplot: financiación, rondas y edad según supervivencia", y=1.02)
plt.show()

## 9. ¿Es una relación real o casualidad?

Una diferencia en los datos puede ser **real** o puede ser **casualidad**. Para comprobarlo usamos tres pruebas estadísticas sencillas.

### 9.1 Chi-cuadrado y V de Cramér

El **chi-cuadrado** mide si los 4 grupos de financiación tienen tasas de supervivencia distintas. Si el valor es **mayor que 7,81**, la diferencia entre grupos es real con un 95 % de confianza.

La **V de Cramér** dice **cómo de grande** es la relación, de 0 a 1:

- menos de 0,10: muy pequeña
- entre 0,10 y 0,30: pequeña o moderada
- más de 0,30: grande

In [ ]:
ct = pd.crosstab(dff["funding_quartile"], dff["survived"]).to_numpy()
esperado = ct.sum(axis=1, keepdims=True) * ct.sum(axis=0, keepdims=True) / ct.sum()
chi2 = ((ct - esperado) ** 2 / esperado).sum()
v_cramer = np.sqrt(chi2 / (ct.sum() * (min(ct.shape) - 1)))
print(f"Chi2 = {chi2:.1f} (3 gl; el valor crítico al 5% es 7.81) | V de Cramér = {v_cramer:.3f}")

### 9.2 Prueba de permutaciones

Aquí hacemos un experimento: **mezclamos al azar** las etiquetas "sobrevive / cierra" miles de veces y vemos qué diferencia de dinero sale **solo por casualidad**. Después comparamos con la diferencia real.

Si la diferencia real es mucho mayor que las del azar, el **p-valor** sale muy pequeño. Un p-valor **menor que 0,05** significa que es muy poco probable que sea casualidad.

In [ ]:
rng = np.random.default_rng(42)
x = dff["log_funding"].to_numpy()
y = dff["survived"].to_numpy()
obs = np.median(x[y == 1]) - np.median(x[y == 0])
perm = np.empty(2000)
for i in range(len(perm)):
    yp = rng.permutation(y)
    perm[i] = np.median(x[yp == 1]) - np.median(x[yp == 0])
p_perm = (np.sum(np.abs(perm) >= abs(obs)) + 1) / (len(perm) + 1)
print(f"Diferencia de medianas (log10 USD): {obs:.3f} | p (permutaciones) = {p_perm:.4f}")

### 9.3 Diferencia entre el grupo que más y el que menos dinero recibió

Calculamos cuántos **puntos porcentuales** más sobreviven las startups del grupo Q4 (más dinero) que las del Q1 (menos dinero). Como esta cifra viene de una muestra, damos también un **intervalo de confianza al 95 %**: un rango donde casi seguro está el valor real. Si el rango **no incluye el 0**, la diferencia es real.

In [ ]:
q = dff["funding_quartile"]
s4 = dff.loc[q == "Q4 (más)", "survived"].to_numpy()
s1 = dff.loc[q == "Q1 (menos)", "survived"].to_numpy()
dif_q = np.array([rng.choice(s4, len(s4)).mean() - rng.choice(s1, len(s1)).mean() for _ in range(2000)])
lo, hi = np.percentile(dif_q, [2.5, 97.5])
print(f"Supervivencia Q4 - Q1: {s4.mean() - s1.mean():+.1%} (IC95 %: {lo:+.1%} a {hi:+.1%})")

**Un aviso importante:** como tenemos muchas startups, casi cualquier diferencia sale "significativa". Por eso hay que mirar sobre todo **el tamaño** de la diferencia (V de Cramér y puntos porcentuales) y no solo el p-valor.

## 10. ¿Importa la financiación aunque tengamos en cuenta otras cosas?

Hasta ahora hemos comparado de una en una. Pero una startup puede sobrevivir por muchos motivos a la vez: su edad, su número de rondas, su país... Para separar el efecto de cada cosa usamos una **regresión logística**, que es un modelo que calcula la **probabilidad de sobrevivir** teniendo en cuenta varias variables a la vez.

El resultado más fácil de leer es el **OR** (*odds ratio*):

- **OR mayor que 1**: esa variable ayuda a sobrevivir (o a ser comprada).
- **OR igual a 1**: no cambia nada.
- **OR menor que 1**: esa variable va con menos supervivencia.

Para `log_funding`, el OR dice cuánto se multiplican las opciones de sobrevivir **por cada 10 veces más dinero**, manteniendo todo lo demás igual. Si el rango `OR_ic95_inf` a `OR_ic95_sup` **no incluye el 1**, el efecto es real.

In [ ]:
def logit_numpy(X, y, n_iter=100, tol=1e-8):
    """Regresión logística hecha a mano con numpy. Devuelve los coeficientes y sus errores."""
    X = np.column_stack([np.ones(len(X)), X])
    beta = np.zeros(X.shape[1])
    for _ in range(n_iter):
        p = 1 / (1 + np.exp(-np.clip(X @ beta, -30, 30)))
        W = p * (1 - p)
        paso = np.linalg.solve(X.T @ (X * W[:, None]), X.T @ (y - p))
        beta = beta + paso
        if np.abs(paso).max() < tol:
            break
    p = 1 / (1 + np.exp(-np.clip(X @ beta, -30, 30)))
    cov = np.linalg.inv(X.T @ (X * (p * (1 - p))[:, None]))
    return beta, np.sqrt(np.diag(cov))


m = dff.dropna(subset=["age", "funding_rounds"]).copy()
vars_x = ["log_funding", "funding_rounds", "age", "us"]
for objetivo in ["survived", "acquired"]:
    beta, se = logit_numpy(m[vars_x].to_numpy(float), m[objetivo].to_numpy(float))
    res = pd.DataFrame({"coef": beta, "se": se, "z": beta / se, "OR": np.exp(beta),
                        "OR_ic95_inf": np.exp(beta - 1.96 * se),
                        "OR_ic95_sup": np.exp(beta + 1.96 * se)},
                       index=["const"] + vars_x)
    print(f"\n=== {objetivo} (n={len(m):,}) ===")
    display(res)

## 11. Los mercados

Ahora miramos por **sectores**. Nos quedamos con los 15 mercados que más startups tienen, porque en los mercados pequeños hay muy pocos datos para sacar conclusiones.

### 11.1 Resumen por mercado

In [ ]:
TOP = 15
dm = dff[dff["market"] != "unknown"]            # con financiación y mercado conocidos
top = dm["market"].value_counts().head(TOP).index
dt = dm[dm["market"].isin(top)]

res_m = (dt.groupby("market")
         .agg(n=("survived", "size"),
              financiacion_mediana=("funding_total_usd", "median"),
              financiacion_total=("funding_total_usd", "sum"),
              supervivencia=("survived", "mean"),
              adquisicion=("acquired", "mean"))
         .sort_values("n", ascending=False))
display(res_m.style.format({"financiacion_mediana": "{:,.0f}", "financiacion_total": "{:,.0f}",
                            "supervivencia": "{:.1%}", "adquisicion": "{:.1%}"}))

### 11.2 ¿Cómo se reparten las startups entre mercados?

A la izquierda, el número de startups de cada mercado. A la derecha, el mismo reparto en porcentaje (los 8 más grandes y el resto agrupado en "otros").

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
dm["market"].value_counts().head(TOP).sort_values().plot.barh(ax=ax[0], color="steelblue")
ax[0].set_title(f"Nº de startups por mercado (top {TOP})"); ax[0].set_xlabel("Startups")

tarta = dm["market"].value_counts().head(8)
tarta["otros"] = len(dm) - tarta.sum()
ax[1].pie(tarta, labels=tarta.index, autopct="%1.0f%%", startangle=90,
          colors=sns.color_palette("Set3", len(tarta)))
ax[1].set_title("Reparto de startups (top 8 + otros)")
plt.tight_layout(); plt.show()

### 11.3 ¿Qué mercados reciben más financiación?

Hay dos formas de medirlo y dicen cosas distintas:

- **Financiación total** (izquierda): todo el dinero que recibe el mercado. Los mercados con **muchas startups** salen arriba aunque cada una reciba poco.
- **Financiación mediana** (derecha): lo que recibe una startup **típica** del mercado. Es la mejor medida para comparar mercados de forma justa.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
res_m["financiacion_total"].div(1e9).sort_values().plot.barh(ax=ax[0], color="teal")
ax[0].set_title("Financiación TOTAL por mercado (miles de millones USD)")
res_m["financiacion_mediana"].div(1e6).sort_values().plot.barh(ax=ax[1], color="darkorange")
ax[1].set_title("Financiación MEDIANA por startup (millones USD)")
plt.tight_layout(); plt.show()

### 11.4 ¿Qué mercados sobreviven más?

Cada barra es el porcentaje de startups del mercado que **no han cerrado**. La línea roja es la media de todas las startups: las barras que la superan son mercados que sobreviven **más que la media**.

In [ ]:
orden_sup = res_m["supervivencia"].sort_values().index
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=res_m.reset_index(), y="market", x="supervivencia", order=orden_sup,
            hue="market", palette="viridis", legend=False, ax=ax)
ax.axvline(dff["survived"].mean(), color="red", ls="--", label="media global")
ax.set_title("Tasa de supervivencia por mercado"); ax.set_xlabel("Proporción que sigue viva")
ax.legend(); plt.show()

### 11.5 ¿Los mercados con más dinero sobreviven más?

Cada círculo es un mercado (más grande = más startups). Si los círculos suben hacia la derecha y la línea roja sube, los mercados que financian más a sus startups también son los que más sobreviven.

Con solo unos 15 mercados, este dato es orientativo: pocos puntos dan una conclusión menos fiable.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=res_m, x=np.log10(res_m["financiacion_mediana"]), y="supervivencia",
                size="n", sizes=(60, 600), legend=False, ax=ax, color="steelblue")
for mkt, r in res_m.iterrows():
    ax.annotate(mkt, (np.log10(r["financiacion_mediana"]), r["supervivencia"]), fontsize=8,
                xytext=(4, 4), textcoords="offset points")
sns.regplot(x=np.log10(res_m["financiacion_mediana"]), y=res_m["supervivencia"],
            scatter=False, ci=None, color="red", ax=ax)
ax.set_xlabel("log10 financiación mediana del mercado (USD)")
ax.set_ylabel("Tasa de supervivencia")
ax.set_title("Mercados: financiación mediana vs supervivencia (tamaño = nº startups)")
plt.show()

rho_mercados = res_m["financiacion_mediana"].rank().corr(res_m["supervivencia"].rank())
print(f"Spearman a nivel de mercado (n={len(res_m)}): {rho_mercados:.2f}")

### 11.6 Dentro de cada mercado

Para comprobar que la relación no es un truco de mezclar mercados distintos, miramos **dentro de cada mercado** (los 10 más grandes) y por grupo de financiación (Q1 a Q4). Si en casi todas las filas el color se oscurece hacia la derecha, la financiación se relaciona con sobrevivir en **todos** los mercados.

In [ ]:
top10 = res_m.head(10).index
hm = (dt[dt["market"].isin(top10)]
      .pivot_table(index="market", columns="funding_quartile", values="survived",
                   aggfunc="mean", observed=True))
plt.figure(figsize=(8, 6))
sns.heatmap(hm, annot=True, fmt=".0%", cmap="YlGnBu", vmin=0.3, vmax=1)
plt.title("Supervivencia por mercado y cuartil de financiación")
plt.ylabel(""); plt.xlabel("")
plt.show()

## 12. Resumen de la relación entre financiación y supervivencia

Juntamos en una sola celda las pistas principales:

1. La **correlación** entre dinero recibido y supervivencia.
2. La **diferencia** entre el grupo que más dinero recibió (Q4) y el que menos (Q1).
3. En cuántos **mercados** la relación va en el mismo sentido.
4. Si se mantiene en cada **grupo de edad** (cohorte).

La última línea imprime un veredicto automático con los números de **tus** datos.

In [ ]:
r_start = dff["log_funding"].rank().corr(dff["survived"].rank())      # Spearman a nivel startup
q = dff["funding_quartile"]
dif = dff.loc[q == "Q4 (más)", "survived"].mean() - dff.loc[q == "Q1 (menos)", "survived"].mean()

por_mercado = (dt.groupby("market")
               .apply(lambda g: g["log_funding"].rank().corr(g["survived"].rank()), include_groups=False))
pos = (por_mercado > 0).mean()

coh = dff.groupby("cohorte", observed=True).apply(
    lambda g: g["log_funding"].rank().corr(g["survived"].rank()), include_groups=False)

print(f"Spearman log_funding vs survived (startups): {r_start:.3f}")
print(f"Supervivencia Q4 - Q1: {dif:+.1%}")
print(f"Mercados con relación positiva: {pos:.0%} de los {len(por_mercado)} analizados")
print("Spearman por cohorte de fundación:\n", coh.round(3).to_string())

fuerza = ("fuerte" if abs(r_start) >= 0.3 else "moderada" if abs(r_start) >= 0.15
          else "débil" if abs(r_start) >= 0.05 else "prácticamente nula")
sentido = "positiva" if r_start > 0 else "negativa"
print(f"\nVEREDICTO: relación {sentido} y {fuerza} entre financiación y supervivencia; "
      f"el cuartil más financiado tiene {dif * 100:+.1f} puntos porcentuales de supervivencia respecto al menos financiado; "
      f"el patrón {'se repite' if pos >= 0.8 else 'es mixto' if pos >= 0.5 else 'NO se repite'} "
      f"entre mercados ({pos:.0%} con signo positivo).")
print("Recuerda: es asociación, no causalidad (las startups más prometedoras reciben más dinero y además sobreviven más).")

## 13. Conclusiones

> **Cómo completar esta parte:** sustituye cada `[...]` por el número o la frase que has obtenido al ejecutar las celdas de arriba. Las guías entre paréntesis te dicen de dónde sale cada dato.

**Respuesta a la pregunta principal.** En nuestros datos, las startups que reciben más financiación **[sobreviven más / no sobreviven más]** que las que reciben menos. (Mira el *VEREDICTO* de la sección 12.)

**Las cifras que lo apoyan:**

- Entre el grupo que más dinero recibió (Q4) y el que menos (Q1) hay **[...] puntos porcentuales** de diferencia en supervivencia. (Sección 9.3.)
- La correlación de Spearman entre dinero y supervivencia es **[...]**, es decir, una relación **[débil / moderada / fuerte]**. (Sección 12.)
- La diferencia **[es / no es]** casualidad: p-valor **[...]** en la prueba de permutaciones. (Sección 9.2.)
- Al tener en cuenta edad, rondas y país a la vez, el OR de `log_funding` es **[...]**, con un rango que **[incluye / no incluye]** el 1. (Sección 10.)

**Sobre los mercados:**

- Los mercados con más startups son **[...]**. (Sección 11.2.)
- Los que reciben más dinero por startup (mediana) son **[...]**. (Sección 11.3.)
- Los que más sobreviven son **[...]** y los que menos son **[...]**. (Sección 11.4.)
- La relación entre dinero y supervivencia **[se repite / no se repite]** en los distintos mercados. (Secciones 11.6 y 12.)

**Sobre el éxito (startups compradas):** la financiación **[también / no]** se relaciona con que una startup sea comprada. (Sección 7.2 y regresión de `acquired`.)

## 14. Limitaciones: lo que este trabajo NO puede decir

1. **Relación no es causa.** Que las startups con más dinero sobrevivan más **no demuestra** que el dinero sea el motivo. Puede ser que las startups más prometedoras consigan más dinero **y además** sobrevivan más por ser buenas.
2. **Los datos son antiguos.** Llegan solo hasta alrededor de 2014. Las startups muy jóvenes han tenido poco tiempo para cerrar, y eso puede hacer que parezcan sobrevivir más.
3. **Solo vemos a quien queda registrado.** Las startups que cierran muy pronto y sin financiación pueden no aparecer, lo que se llama *sesgo de supervivencia*.
4. **Falta de datos no es "cero dinero".** Las startups sin dato de financiación se dejaron fuera del análisis de dinero.
5. **"Operating" mezcla casos muy distintos.** Dentro de las startups que "siguen funcionando" hay empresas que crecen y empresas casi paradas.
6. **El éxito es una aproximación.** Medir el éxito solo como "ser comprada" deja fuera otros éxitos, como salir a bolsa o ser rentable.

## 15. Cierre

Este análisis muestra **si existe relación** entre financiación y supervivencia, y **qué tan fuerte es** en los datos. Para saber si el dinero **causa** que una startup sobreviva haría falta otro tipo de estudio, por ejemplo comparar startups muy parecidas donde solo cambie el dinero recibido.